In [1]:
import nltk
import pickle
import pandas as pd
import os
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

In [ ]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_email(message):
    # message is a valid string
    if not isinstance(message, str):
        return ""

    # remove extra spaces
    message = ' '.join(message.split())

    # tokenize word
    words = word_tokenize(message)
    # ada uppercase kita ganti jadi lowercase
    words = [w.lower() for w in words if w.isalpha() and w not in stop_words]

    # apply stemming
    words = [stemmer.stem(w) for w in words]

    return ' '.join(words)

In [12]:
def load_train_model(model_file='model.pickle'):
    # model.pickle is found
    if os.path.exists(model_file):
        with open(model_file, 'rb') as f:
            modelNB, tfidf, email_matrix = pickle.load(f)
            print("Model loaded from model.pickle")
        return modelNB, tfidf, email_matrix
    # model.pickle is not found
    else:
        # read dataset
        df = pd.read_csv("combined_data.csv")
        category = df['label']
        message = df['text']

        # preprocess email column
        message = message.apply(preprocess_email)

        # feature extraction with tfidf
        tfidf = TfidfVectorizer(max_features=5000)
        x = tfidf.fit_transform(message)
        y = category

        # train using naive bayes
        modelNB = MultinomialNB()
        modelNB.fit(x, y)

        # getting tfidf matrix result
        email_matrix = tfidf.transform(message)

        # Make predictions
        y_pred = modelNB.predict(x)

        # Calculate accuracy
        acc = accuracy_score(y_pred, y)
        print(f"Model accuracy: {acc:.2f}")

        # save model
        with open(model_file, 'wb') as f:
            pickle.dump((modelNB, tfidf, email_matrix), f)
            print("Model created successfully!")
            return modelNB, tfidf, email_matrix

In [13]:
def write_email(tfidf):
    email = input("Enter email message (min: 10 words): ")
    while len(email.split()) < 10:
        email = input("Email message must be at least 10 words: ")

    # preprocess the sentence
    email_preprocess = preprocess_email(email)

    # vectorize using tfidf
    email_vectorize = tfidf.transform([email_preprocess])

    # load model to classify the email message (ham or spam)
    model, _, _ = load_train_model()
    # Classify the email
    prediction = model.predict(email_vectorize)
    print(f"Email message: {email}")
    print(f"Result: {prediction[0]}")

In [18]:
model, tfidf, movie_review_matrix = load_train_model()

Model accuracy: 0.97
Model created successfully!


In [23]:
write_email(tfidf)

Model loaded from model.pickle
Email message: You are eligible to receive a FREE iPhone 15 Pro with no purchase required. To claim your prize, simply confirm your delivery details by clicking the link below within the next 2 hours
Result: 1
